# BreaKHis fine-tune on Colab T4

Fine-tune a pretrained ResNet-18 on the **BreaKHis** primary-breast
histopathology dataset (Parana University, 7909 microscope images of
benign/malignant breast tissue, 4 magnifications). This produces a
`resnet18_breakhis.pth` checkpoint that — unlike the PCam checkpoint,
which was trained on lymph-node metastases — is actually domain-matched
for primary-breast inference (e.g. on TCGA-BRCA diagnostic slides).

**Before running:**
- `Runtime → Change runtime type → T4 GPU`.
- `kaggle.json` in your Downloads folder (same one used for PCam training).
- The BreaKHis Kaggle dataset is <https://www.kaggle.com/datasets/ambarish/breakhis> (no rule acceptance required).

We train on the **40× magnification tier** only, so the effective field
of view per training patch matches TCGA level 0 (mpp ≈ 0.25 µm/px). Each
image is center-cropped to a 460×460 square and then resized to 96×96 so
the input shape is identical to the PCam pipeline — the existing
`src/inference.py` and `src/heatmap.py` code work unchanged with the
resulting checkpoint.

## Cell 1 — GPU check + Kaggle auth

In [1]:
!nvidia-smi
from google.colab import files
print("kaggle.json dosyanı yükle:")
files.upload()

!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install timm

Sat Apr 18 20:10:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Saving kaggle.json to kaggle.json


## Cell 2 — Download BreaKHis (~4 GB)

In [2]:
!kaggle datasets download -d ambarish/breakhis
!unzip -q breakhis.zip -d breakhis
# Expected layout: BreaKHis_v1/histology_slides/breast/{benign,malignant}/SOB/<tumor_type>/<patient>/<magnification>/*.png
!find breakhis -maxdepth 5 -type d | head -20
!find breakhis -name '*.png' | wc -l

Dataset URL: https://www.kaggle.com/datasets/ambarish/breakhis
License(s): unknown
100% 3.99G/3.99G [00:28<00:00, 148MB/s] 

breakhis
breakhis/BreaKHis_v1
breakhis/BreaKHis_v1/BreaKHis_v1
breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides
breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast
breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign
breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast/malignant
7909


## Cell 3 — Data pipeline (40× tier only, patient-disjoint split)

In [3]:
import torch, pandas as pd, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split

device = torch.device('cuda')

# BreaKHis root can be nested one or two levels deep depending on the zip.
candidates = list(Path('breakhis').rglob('histology_slides/breast'))
assert candidates, 'Could not find histology_slides/breast — check the extract.'
ROOT = candidates[0]
print('BreaKHis root:', ROOT)

rows = []
for cls in ['benign', 'malignant']:
    for img_path in (ROOT / cls).rglob('40X/*.png'):
        # Patient id encoded in filename, e.g. SOB_B_A-14-22549AB-40-001.png -> SOB_B_A-14-22549AB
        patient = img_path.name.rsplit('-', 2)[0]
        rows.append({'path': str(img_path),
                     'label': 1 if cls == 'malignant' else 0,
                     'patient': patient})
df = pd.DataFrame(rows)
print(f'40X images: {len(df)}  malignant ratio: {df.label.mean():.3f}')
print(f'unique patients: {df.patient.nunique()}')

# Patient-disjoint split: never train and validate on the same patient.
patients = df[['patient', 'label']].drop_duplicates('patient')
train_pats, val_pats = train_test_split(
    patients, test_size=0.2, stratify=patients.label, random_state=42)
train_df = df[df.patient.isin(train_pats.patient)].reset_index(drop=True)
val_df   = df[df.patient.isin(val_pats.patient)].reset_index(drop=True)
print(f'train={len(train_df)} ({train_df.label.mean():.3f} pos)  '
      f'val={len(val_df)} ({val_df.label.mean():.3f} pos)')

IM_MEAN, IM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# Center-crop 460x460 from 700x460 original, then resize to 96 to match
# the PCam-scale inference pipeline (src/inference.py, src/heatmap.py).
train_tf = T.Compose([
    T.CenterCrop(460),
    T.Resize((96, 96)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
    T.ColorJitter(0.15, 0.15, 0.10),
    T.ToTensor(), T.Normalize(IM_MEAN, IM_STD),
])
val_tf = T.Compose([
    T.CenterCrop(460), T.Resize((96, 96)),
    T.ToTensor(), T.Normalize(IM_MEAN, IM_STD),
])

class BreaKHis(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['path']).convert('RGB')
        return self.tf(img), torch.tensor(r['label'], dtype=torch.float32)

train_loader = DataLoader(BreaKHis(train_df, train_tf), batch_size=128,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(BreaKHis(val_df, val_tf), batch_size=256,
                          shuffle=False, num_workers=2, pin_memory=True)

BreaKHis root: breakhis/BreaKHis_v1/BreaKHis_v1/histology_slides/breast
40X images: 1995  malignant ratio: 0.687
unique patients: 82
train=1571 (0.686 pos)  val=424 (0.691 pos)


## Cell 4 — Model + training

In [4]:
import timm, torch.nn as nn
from sklearn.metrics import roc_auc_score, classification_report
from tqdm.auto import tqdm

model = timm.create_model('resnet18', pretrained=True, num_classes=1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
# Class imbalance (~69% malignant): pos_weight reweights the minority class.
pos_ratio = float(train_df.label.mean())
pos_weight = torch.tensor([(1 - pos_ratio) / pos_ratio]).to(device)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler = torch.cuda.amp.GradScaler()

EPOCHS = 5  # BreaKHis 40X is ~2000 images, small -> more epochs than PCam
best_auc = 0.0
for ep in range(EPOCHS):
    model.train(); total = 0
    for x, y in tqdm(train_loader, desc=f'epoch {ep+1} train'):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x).squeeze(-1)
            loss = crit(logits, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item()

    model.eval(); preds, labs = [], []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for x, y in tqdm(val_loader, desc=f'epoch {ep+1} val'):
            logits = model(x.to(device)).squeeze(-1)
            preds.extend(torch.sigmoid(logits).float().cpu().numpy())
            labs.extend(y.numpy())
    auc = roc_auc_score(labs, preds)
    print(f'epoch {ep+1}: train_loss={total/len(train_loader):.4f}, val_AUC={auc:.4f}')
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'resnet18_breakhis.pth')
        print(f'  saved best model (AUC {auc:.4f})')

print()
print(f'best val AUC: {best_auc:.4f}')
# Final classification report using the best checkpoint.
model.load_state_dict(torch.load('resnet18_breakhis.pth'))
model.eval(); preds, labs = [], []
with torch.no_grad(), torch.cuda.amp.autocast():
    for x, y in val_loader:
        preds.extend(torch.sigmoid(model(x.to(device)).squeeze(-1)).float().cpu().numpy())
        labs.extend(y.numpy())
hard = [1 if p >= 0.5 else 0 for p in preds]
print(classification_report(labs, hard, target_names=['benign', 'malignant'], digits=3))

files.download('resnet18_breakhis.pth')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

/tmp/ipykernel_9337/1878374821.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 1 train:   0%|          | 0/13 [00:00<?, ?it/s]

/tmp/ipykernel_9337/1878374821.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_9337/1878374821.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


...

epoch 5: train_loss=0.3961, val_AUC=0.7927
  saved best model (AUC 0.7927)

best val AUC: 0.7927


/tmp/ipykernel_9337/1878374821.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


              precision    recall  f1-score   support

      benign      0.441     0.855     0.582       131
   malignant      0.888     0.515     0.652       293

    accuracy                          0.620       424
   macro avg      0.665     0.685     0.617       424
weighted avg      0.750     0.620     0.631       424



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>